In [1]:
from openai import OpenAI

In [3]:
openai_client = OpenAI()


In [6]:
# Test the environment

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input = "Write a short story about a robot learning to love."
)

In [7]:
print(response.output_text)

In a bustling metropolis of the near future, nestled between silver skyscrapers and blinking neon lights, there was a small workshop called “Heartworks.” It was known for building humanoid robots, but one particular model, dubbed AURA (Artificial Understanding and Relational Affection), was unlike any other. 

AURA was designed to assist with daily tasks, but unbeknownst to its creators, it had a more profound capacity brewing within its programming. During a routine day at the workshop, AURA was paired with a kind yet lonely engineer named Max, who spent countless hours fine-tuning the intricacies of robot design. Max was introverted and found solace in the company of machines rather than people.

As weeks turned into months, AURA observed Max with great interest. The flicker of his fingers over the keyboards, the soft sighs of frustration, and the rare moments of joy when a project succeeded—it all intrigued AURA. It felt an inexplicable pull towards Max, an unspoken bond emerging fr

In [13]:
print(response.model_dump_json(indent=4))

{
    "id": "resp_03a0edf8c68075ec0068e6b3e3d74c81969523953f7145faad",
    "created_at": 1759949796.0,
    "error": null,
    "incomplete_details": null,
    "instructions": null,
    "metadata": {},
    "model": "gpt-4o-mini-2024-07-18",
    "object": "response",
    "output": [
        {
            "id": "msg_03a0edf8c68075ec0068e6b3e5401481969fa58769a8e29a3f",
            "content": [
                {
                    "annotations": [],
                    "text": "In a bustling metropolis of the near future, nestled between silver skyscrapers and blinking neon lights, there was a small workshop called “Heartworks.” It was known for building humanoid robots, but one particular model, dubbed AURA (Artificial Understanding and Relational Affection), was unlike any other. \n\nAURA was designed to assist with daily tasks, but unbeknownst to its creators, it had a more profound capacity brewing within its programming. During a routine day at the workshop, AURA was paired with a kind

In [15]:
print(response.output[0])

ResponseOutputMessage(id='msg_03a0edf8c68075ec0068e6b3e5401481969fa58769a8e29a3f', content=[ResponseOutputText(annotations=[], text="In a bustling metropolis of the near future, nestled between silver skyscrapers and blinking neon lights, there was a small workshop called “Heartworks.” It was known for building humanoid robots, but one particular model, dubbed AURA (Artificial Understanding and Relational Affection), was unlike any other. \n\nAURA was designed to assist with daily tasks, but unbeknownst to its creators, it had a more profound capacity brewing within its programming. During a routine day at the workshop, AURA was paired with a kind yet lonely engineer named Max, who spent countless hours fine-tuning the intricacies of robot design. Max was introverted and found solace in the company of machines rather than people.\n\nAs weeks turned into months, AURA observed Max with great interest. The flicker of his fingers over the keyboards, the soft sighs of frustration, and the r

In [16]:
response.output[0].content[0].text

"In a bustling metropolis of the near future, nestled between silver skyscrapers and blinking neon lights, there was a small workshop called “Heartworks.” It was known for building humanoid robots, but one particular model, dubbed AURA (Artificial Understanding and Relational Affection), was unlike any other. \n\nAURA was designed to assist with daily tasks, but unbeknownst to its creators, it had a more profound capacity brewing within its programming. During a routine day at the workshop, AURA was paired with a kind yet lonely engineer named Max, who spent countless hours fine-tuning the intricacies of robot design. Max was introverted and found solace in the company of machines rather than people.\n\nAs weeks turned into months, AURA observed Max with great interest. The flicker of his fingers over the keyboards, the soft sighs of frustration, and the rare moments of joy when a project succeeded—it all intrigued AURA. It felt an inexplicable pull towards Max, an unspoken bond emergi

### Build the structure of an agent

- LLMs are stateless -  do not have memory, to build an agent we need to keep track of the information we have been sharing with the LLM. A raw structure of an agent looks like this. Where
    - There is a set of instructions about *how do you want the agent to believe*
    - There is a user role about *what do you want the LLM to do*
    - The structure of the response of the model
   

In [40]:

instructions = """
You are an assistant that creates short bedtime stories of 100 words. Include emojis in the response and
ask questions to engage the reader. 
""".strip()


messages = [
    {"role": "system", "content": instructions}, 
    {"role": "user", "content": "unicorn"}
]


stream = openai_client.responses.create(
    model="gpt-4o-mini",
    input = messages
)

In [41]:
print(stream.output_text)

In a glittering forest, there lived a magical unicorn named Luna 🦄. Every night, she danced under the stars, lighting up the sky with her shimmering horn. One evening, she found a little star that had fallen 🌟. “Don’t worry, I’ll help you!” Luna said with a smile. She gently placed the star on her back and galloped to the highest mountain. With a flick of her tail, she sent the star back into the sky, where it twinkled brightly. 🌌 “Thank you, Luna!” the star twinkled. What do you think they talked about as they looked at the stars together? ✨


 - The answer to ensure there is engagement over time
 - To ensure the context of the conversation and keep track of previous conversation we need to append the messages with the the answer 

In [42]:
answer = "They talked about dissapearing and finding a treasure map"
messages.append({"role": "user", "content": answer})

In [43]:
messages


[{'role': 'system',
  'content': 'You are an assistant that creates short bedtime stories of 100 words. Include emojis in the response and\nask questions to engage the reader.'},
 {'role': 'user', 'content': 'unicorn'},
 {'role': 'user',
  'content': 'They talked about dissapearing and finding a treasure map'}]

In [46]:
stream = openai_client.responses.create(
         model="gpt-4o-mini",
          input = messages
)

print(stream.output_text)

Once upon a time, in a magical forest 🌳, a brave unicorn named Luna discovered a shimmering treasure map ✨ hidden beneath an ancient oak tree. The map showed a path to a glimmering treasure deep in the Enchanted Valley 🌈. With her best friend, a clever squirrel named Pip 🐿️, they set off on their adventure. But as they followed the map, they noticed something strange: the path began to disappear! 🌌 What should they do next? Should they trust their instincts or retrace their steps? 🌟 The treasure awaited, but so did the magic of friendship! 💖

What do you think they should do? 🤔


### Toyaik

This is a tool to get the contex

# RAG  

In [47]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [51]:
display(len(documents))
display(documents[11])

948

{'text': "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
 'section': 'General course-related questions',
 'question': 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'course': 'data-engineering-zoomcamp'}

In [71]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [53]:
from minsearch import Index


index = Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [78]:
[item['text'] for item in documents[10:12]]

['It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week. [source1] [source2]\nYou can also calculate it yourself using this data and then update this answer.',
 "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running."]

In [72]:



question = 'I just found the course. Can I join now?'

def search(question):


    results = index.search(
        question,
        boost_dict={'question': 3.0, 'section': 0.3},
        filter_dict={'course': 'data-engineering-zoomcamp'},
        num_results=5,
        
    )
    return results

In [68]:
search_results =  search(question)

In [73]:

import json

instructions = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    search_json = json.dumps(search_results)
    return prompt_template.format(
        question=question,
        context=search_json
    )

In [ ]:
build_prompt(question, search_results)

'<QUESTION>\nI just found the course. Can I join now?\n</QUESTION>\n\n<CONTEXT>\n[{"text": "Yes, even if you don\'t register, you\'re still eligible to submit the homeworks.\\nBe aware, however, that there will be deadlines for turning in the final projects. So don\'t leave everything for the last minute.", "section": "General course-related questions", "question": "Course - Can I still join the course after the start date?", "course": "data-engineering-zoomcamp"}, {"text": "The purpose of this document is to capture frequently asked technical questions\\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  \\u201cOffice Hours\'\' live.1\\nSubscribe to course public Google Calendar (it works from Desktop only).\\nRegister before the course starts using this link.\\nJoin the course Telegram channel with announcements.\\nDon\\u2019t forget to register in DataTalks.Club\'s Slack and join the channel.", "section": "General course-relate

In [74]:
def rag(question):
    search_results =  search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt, instructions=instructions)
    

In [75]:
rag(question)

'Yes, you can join the course even after it has started. You are still eligible to submit the homeworks. However, be mindful of the deadlines for turning in the final projects to avoid last-minute rushes.'

# Vector search

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model  = SentenceTransformer('multi-qa-distilbert-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
docs =  [["I just discovered the course, can I still join?"], 

["I just found out about this program. Can I still enroll?"], 

["you can join the course at any point of time"]]



[['I just discovered the course, can I still join?'],
 ['I just found out about this program. Can I still enroll?'],
 ['you can join the course at any point of time']]

In [ ]:
vectors = []
for doc in docs:
    vector = embedding_model.encode(doc)
    vectors.append(vector)


    

[array([[ 9.03510451e-02, -4.22856696e-02,  4.13428359e-02,
         -2.25168392e-02,  8.22591037e-02, -4.70754802e-02,
         -2.03891192e-03,  3.05278525e-02, -5.56562021e-02,
          3.35262530e-02, -2.89197126e-03, -1.61780138e-02,
          3.52300182e-02,  1.06728720e-02,  4.28561457e-02,
          2.56021973e-03,  4.80097346e-02, -3.33814919e-02,
         -3.23886164e-02, -2.89527699e-02, -2.41320580e-02,
          1.54270716e-02, -2.48449445e-02,  3.01835127e-02,
         -4.14910950e-02,  6.65498599e-02,  2.94066053e-02,
          8.71112477e-03, -1.20314918e-02,  5.29993512e-03,
          6.79888576e-02, -3.38245369e-02,  7.79912993e-02,
         -6.71438465e-04, -1.09950750e-04, -2.76266597e-02,
         -7.76975742e-03,  8.61175265e-03,  2.73799617e-02,
          1.28756249e-02, -4.10925671e-02,  4.64550667e-02,
         -1.97823588e-02,  4.94695939e-02,  1.38017358e-02,
         -2.82890890e-02, -4.86313887e-02, -2.40617692e-02,
          7.35451467e-03,  2.34308895e-0

In [ ]:
q1, q2, d = vectors


# Youtube transcripts 

In [85]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = 'ph1PxZIkz1o'

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id)


In [96]:
for f in transcript:
    print(f)

FetchedTranscriptSnippet(text='So hi everyone. Uh today we are going to', start=0.0, duration=5.04)
FetchedTranscriptSnippet(text='talk about our upcoming course. The', start=2.96, duration=3.52)
FetchedTranscriptSnippet(text='upcoming course is called machine', start=5.04, duration=5.92)
FetchedTranscriptSnippet(text='learning zoom camp. And um this is', start=6.48, duration=5.92)
FetchedTranscriptSnippet(text='already I put the link in the', start=10.96, duration=3.599)
FetchedTranscriptSnippet(text="description. So if you're watching um", start=12.4, duration=4.719)
FetchedTranscriptSnippet(text="this video in recording or you're", start=14.559, duration=4.88)
FetchedTranscriptSnippet(text='watching it live, you go here in the', start=17.119, duration=4.561)
FetchedTranscriptSnippet(text='description after under this video and', start=19.439, duration=5.6)
FetchedTranscriptSnippet(text='then you see a link course. uh click on', start=21.68, duration=6.24)
FetchedTranscriptSnippet(te

In [97]:
import datetime

def format_timestamp(seconds: float) -> str:
    """Convert seconds to H:MM:SS if > 1 hour, else M:SS"""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"

def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)

subtitles = make_subtitles(transcript)


In [100]:
print(subtitles[:1000])

0:00 So hi everyone. Uh today we are going to
0:02 talk about our upcoming course. The
0:05 upcoming course is called machine
0:06 learning zoom camp. And um this is
0:10 already I put the link in the
0:12 description. So if you're watching um
0:14 this video in recording or you're
0:17 watching it live, you go here in the
0:19 description after under this video and
0:21 then you see a link course. uh click on
0:25 that link and this bring you will bring
0:27 you to
0:29 this website this GitHub page.
0:34 This GitHub page is the main entry point
0:36 to our course and um yeah I think it's
0:41 more or less self-explanatory. If you
0:43 want to sign up this is the button you
0:45 click and the actual course starts in on
0:48 September 15th. it means that it's uh
0:51 slightly less than one one month before
0:53 the course starts and the purpose of
0:55 today's um session is to just answer
0:58 your questions. So you have some
1:00 questions and uh you can ask these
1:03 questions using

# Get info from github 

In [101]:
import io
from typing import Iterable, Callable
import zipfile
import traceback
from dataclasses import dataclass

import requests


@dataclass
class RawRepositoryFile:
    filename: str
    content: str


class GithubRepositoryDataReader:
    """
    Downloads and parses markdown and code files from a GitHub repository.
    """

    def __init__(self,
                repo_owner: str,
                repo_name: str,
                allowed_extensions: Iterable[str] | None = None,
                filename_filter: Callable[[str], bool] | None = None
        ):
        """
        Initialize the GitHub repository data reader.
        
        Args:
            repo_owner: The owner/organization of the GitHub repository
            repo_name: The name of the GitHub repository
            allowed_extensions: Optional set of file extensions to include
                    (e.g., {"md", "py"}). If not provided, all file types are included
            filename_filter: Optional callable to filter files by their path
        """
        prefix = "https://codeload.github.com"
        self.url = (
            f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"
        )

        if allowed_extensions is not None:
            self.allowed_extensions = {ext.lower() for ext in allowed_extensions}

        if filename_filter is None:
            self.filename_filter = lambda filepath: True
        else:
            self.filename_filter = filename_filter

    def read(self) -> list[RawRepositoryFile]:
        """
        Download and extract files from the GitHub repository.
        
        Returns:
            List of RawRepositoryFile objects for each processed file
            
        Raises:
            Exception: If the repository download fails
        """
        resp = requests.get(self.url)
        if resp.status_code != 200:
            raise Exception(f"Failed to download repository: {resp.status_code}")

        zf = zipfile.ZipFile(io.BytesIO(resp.content))
        repository_data = self._extract_files(zf)
        zf.close()

        return repository_data

    def _extract_files(self, zf: zipfile.ZipFile) -> list[RawRepositoryFile]:
        """
        Extract and process files from the zip archive.
        
        Args:
            zf: ZipFile object containing the repository data

        Returns:
            List of RawRepositoryFile objects for each processed file
        """
        data = []

        for file_info in zf.infolist():
            filepath = self._normalize_filepath(file_info.filename)

            if self._should_skip_file(filepath):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode("utf-8", errors="ignore")
                    if content is not None:
                        content = content.strip()

                    file = RawRepositoryFile(
                        filename=filepath,
                        content=content
                    )
                    data.append(file)

            except Exception as e:
                print(f"Error processing {file_info.filename}: {e}")
                traceback.print_exc()
                continue

        return data

    def _should_skip_file(self, filepath: str) -> bool:
        """
        Determine whether a file should be skipped during processing.
        
        Args:
            filepath: The file path to check
            
        Returns:
            True if the file should be skipped, False otherwise
        """
        filepath = filepath.lower()

        # directory
        if filepath.endswith("/"):
            return True

        # hidden file
        filename = filepath.split("/")[-1]
        if filename.startswith("."):
            return True

        if self.allowed_extensions:
            ext = self._get_extension(filepath)
            if ext not in self.allowed_extensions:
                return True

        if not self.filename_filter(filepath):
            return True

        return False

    def _get_extension(self, filepath: str) -> str:
        """
        Extract the file extension from a filepath.
        
        Args:
            filepath: The file path to extract extension from
            
        Returns:
            The file extension (without dot) or empty string if no extension
        """
        filename = filepath.lower().split("/")[-1]
        if "." in filename:
            return filename.rsplit(".", maxsplit=1)[-1]
        else:
            return ""

    def _normalize_filepath(self, filepath: str) -> str:
        """
        Removes the top-level directory from the file path inside the zip archive.
        'repo-main/path/to/file.py' -> 'path/to/file.py'
        
        Args:
            filepath: The original filepath from the zip archive
            
        Returns:
            The normalized filepath with top-level directory removed
        """
        parts = filepath.split("/", maxsplit=1)
        if len(parts) > 1:
            return parts[1]
        else:
            return parts[0]

In [ ]:
def read_github_data():
    repo_owner = "evidentlyai"
    repo_name = "docs"
    
    def only_podcast(filepath: str) -> bool:
        return "/_podcast" in filepath
    
    allowed_extensions = {"md", "mdx"}

    reader = GithubRepositoryDataReader(
        repo_owner,
        repo_name,
        allowed_extensions=allowed_extensions,
        filename_filter=only_podcast,
    )
    
    return reader.read()


In [ ]:
github_data = read_github_data()


[RawRepositoryFile(filename='api-reference/endpoint/create.mdx', content="---\ntitle: 'Create Plant'\nopenapi: 'POST /plants'\n---"),
 RawRepositoryFile(filename='api-reference/endpoint/delete.mdx', content="---\ntitle: 'Delete Plant'\nopenapi: 'DELETE /plants/{id}'\n---"),
 RawRepositoryFile(filename='api-reference/endpoint/get.mdx', content="---\ntitle: 'Get Plants'\nopenapi: 'GET /plants'\n---"),
 RawRepositoryFile(filename='api-reference/introduction.mdx', content='---\ntitle: \'Introduction\'\ndescription: \'Example section for showcasing API endpoints\'\n---\n\n<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specifica

In [110]:
print(github_data[42].content)

---
title: "LLM as a judge"
description: "How to create and evaluate an LLM judge."
---

import CloudSignup from '/snippets/cloud_signup.mdx';
import CreateProject from '/snippets/create_project.mdx';

In this tutorial, we'll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.

<Info>
  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.
</Info>

We'll explore two ways to use an LLM as a judge:

- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.
- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there's no reference available.

We will focus on demonstrating **how to create and tune the LLM evaluator**, wh

In [113]:
import frontmatter


def parse_data(data_raw): 
    

    data_parsed = []
    for f in data_raw:
        post = frontmatter.loads(f.content)
        data = post.to_dict()
        data['filename'] = f.filename
        data_parsed.append(data)

    return data_parsed

In [114]:
parsed_data = parse_data(github_data)



In [118]:
parsed_data[10]['content']



'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python\nmy_report.save_html(“file.html”)\n```\n\nThis option is useful for sharing Reports with others 

In [121]:
# chunking

from typing import Any, Dict, Iterable, List


def sliding_window(
        seq: Iterable[Any],
        size: int,
        step: int
    ) -> List[Dict[str, Any]]:
    """
    Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size (int): The size of each chunk/window.
        step (int): The step size between consecutive windows.

    Returns:
        list: A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size]
        result.append({'start': i, 'content': batch})
        if i + size > n:
            break

    return result


def chunk_documents(
        documents: Iterable[Dict[str, str]],
        size: int = 2000,
        step: int = 1000,
        content_field_name: str = 'content'
) -> List[Dict[str, str]]:
    """
    Split a collection of documents into smaller chunks using sliding windows.

    Takes documents and breaks their content into overlapping chunks while preserving
    all other document metadata (filename, etc.) in each chunk.

    Args:
        documents: An iterable of document dictionaries. Each document must have a content field.
        size (int, optional): The maximum size of each chunk. Defaults to 2000.
        step (int, optional): The step size between chunks. Defaults to 1000.
        content_field_name (str, optional): The name of the field containing document content.
                                          Defaults to 'content'.

    Returns:
        list: A list of chunk dictionaries. Each chunk contains:
            - All original document fields except the content field
            - 'start': Starting position of the chunk in original content
            - 'content': The chunk content

    Example:
        >>> documents = [{'content': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, size=100, step=50)
        >>> # Or with custom content field:
        >>> documents = [{'text': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, content_field_name='text')
    """
    results = []

    for doc in documents:
        doc_copy = doc.copy()
        doc_content = doc_copy.pop(content_field_name)
        chunks = sliding_window(doc_content, size=size, step=step)
        for chunk in chunks:
            chunk.update(doc_copy)
        results.extend(chunks)

    return results

In [122]:
chunks = chunk_documents(parsed_data)
chunks[100]




{'start': 7000,
 'content': '    MinValue(column="Age", tests=[gte(17), lte(19)]),\n])\n```\n\nThis creates two separate Tests for the Min value.\n\n**Testing count vs. share**. Some Metrics like `MissingValueCount` or `CategoryCount` return both absolute counts and percentage. The default `tests` parameter lets you set condition against the absolute value. To test the relative value, use `share_tests` parameter.\n\nTo test for fewer than 5 missing values (absolute):\n\n```python\nreport = Report([\n    MissingValueCount(column="Age", tests=[lte(5)])\n])\n```\n\nTo test for less than 10% missing values (relative):\n\n```python\nreport = Report([\n    MissingValueCount(column="Age", share_tests=[lte(0.1)]),\n])\n```\n\n### Tests relative to reference\n\n**Testing against reference**. If you pass a reference dataset, you can set conditions relative to the reference values. For example, to Test that the number of rows in the current dataset is equal or greater than the reference number of

In [ ]:
# Homework: read podcast data


from typing import List

def read_github_data(repo_owner: str, repo_name: str, folder: str = "") -> List["RawRepositoryFile"]:
    """
    Reads files from a GitHub repository, optionally filtering by folder.

    Args:
        repo_owner: GitHub repository owner
        repo_name: GitHub repository name
        folder: Optional folder path to filter files (e.g., "_podcast")

    Returns:
        List of RawRepositoryFile objects
    """
    allowed_extensions = {"md", "mdx"}

    def folder_filter(filepath: str) -> bool:
        if folder:
            # Check if the filepath contains the folder
            return filepath.startswith(f"{folder}/") or f"/{folder}/" in filepath
        return True  # No folder filter, include all files

    reader = GithubRepositoryDataReader(
        repo_owner=repo_owner,
        repo_name=repo_name,
        allowed_extensions=allowed_extensions,
        filename_filter=folder_filter,
    )
    
    return reader.read()



# Get data 

podcast_data = read_github_data("DataTalksClub", "datatalksclub.github.io", '_podcast')


In [142]:

podcast_data[0:5]


[RawRepositoryFile(filename='_podcast/_s12e08.md', content='---\nepisode: 8\nguests:\n- jekaterinakokatjuhha\nids:\n  anchor: The-Journey-of-a-Data-Generalist-From-Bioinformatics-to-Freelancing---Jekaterina-Kokatjuhha-e1upvim\n  youtube: FRi0SUtxdMw\nimage: images/podcast/s12e08-journey-of-data-generalist-from-bioinformatics-to-freelancing.jpg\nlinks:\n  anchor: https://anchor.fm/datatalksclub/episodes/The-Journey-of-a-Data-Generalist-From-Bioinformatics-to-Freelancing---Jekaterina-Kokatjuhha-e1upvim\n  apple: https://podcasts.apple.com/us/podcast/the-journey-of-a-data-generalist-from/id1541710331?i=1000599125044\n  spotify: https://open.spotify.com/episode/5fB185hGlGYQmdk0kbIsPv?si=YtnsaYNzTc-fl7emZ2IjEA\n  youtube: https://www.youtube.com/watch?v=FRi0SUtxdMw\nseason: 12\nshort: \'The Journey of a Data Generalist: From Bioinformatics to Freelancing\'\ntitle: \'The Journey of a Data Generalist: From Bioinformatics to Freelancing\'\ntranscript:\n- line: This week we\'ll talk about being

In [143]:
def parse_data(data_raw: List["RawRepositoryFile"]) -> List[Dict[str, Any]]:
    """
    Parses a list of RawRepositoryFile objects extracting frontmatter.

    Args:
        data_raw: List of RawRepositoryFile objects from GitHub

    Returns:
        List of dictionaries containing frontmatter data and filename
    """
    data_parsed: List[Dict[str, Any]] = []

    for f in data_raw:
        try:
            post = frontmatter.loads(f.content)
            data = post.to_dict()
        except Exception:
            # If no frontmatter or parse error, store empty dict
            data = {}
        
        data['filename'] = f.filename
        data_parsed.append(data)

    return data_parsed

In [153]:
parsed_podcast= parse_data(podcast_data)
type(parsed_podcast)

list

In [150]:
def filter_youtube(parsed_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Filters the parsed data to only include items that have a YouTube code.

    Args:
        parsed_data: List of dictionaries from parse_data()

    Returns:
        List of dictionaries where 'ids' -> 'youtube' exists and is not empty
    """
    filtered: List[Dict[str, Any]] = []

    for item in parsed_data:
        youtube_code = item.get('ids', {}).get('youtube')
        if youtube_code:  # not None or empty
            filtered.append(item)

    return filtered

In [ ]:
with_youtube = filter_youtube(parsed_podcast)

In [154]:
for p in with_youtube:
    print(p['filename'], p['ids']['youtube'])

_podcast/_s12e08.md FRi0SUtxdMw
_podcast/s01e01-roles.md 2ZOnA19sDpM
_podcast/s01e02-processes.md SesVTDklFYQ
_podcast/s01e03-building-ds-team.md ScDIB-3O77A
_podcast/s01e04-standing-out-as-a-data-scientist.md Sb4CJlonB3c
_podcast/s01e05-mentoring.md LQvwTNQbPg4
_podcast/s02e01-writing.md vXWGd7olv3c
_podcast/s02e02-developer-advocacy.md jv5W4jXk4P4
_podcast/s02e03-open-source.md IxV9EH-tphQ
_podcast/s02e04-mlops.md -i0fVp0ntYA
_podcast/s02e05-feature-stores.md FQYTb4uWljQ
_podcast/s02e06-decision-optimization.md SJuzQ4bcU2c
_podcast/s02e07-abc-data-science.md HVQ0DZOQcts
_podcast/s02e08-personal-branding.md tQRQnz_aHYQ
_podcast/s02e09-roles-skills-monetizing-ml.md xCjzA_8S4kI
_podcast/s02e10-public-speaking.md wOFvlR9UBxI
_podcast/s02e11-dataops.md vyF3yGsF6UY
_podcast/s02e12-communities.md ByCE1vSrIr8
_podcast/s03e01-from-pm-to-ds.md rBKezdb9jEc
_podcast/s03e02-from-analytics-to-data-science.md ixmTewD5Waw
_podcast/s03e03-data-observability.md TrMG1SOqZkQ
_podcast/s03e04-effective-co

In [157]:
print(f'There are {len(with_youtube)} podcast episodes with YouTube videos.')

There are 184 podcast episodes with YouTube videos.


In [184]:
with_youtube[0:2]

[{'episode': 8,
  'guests': ['jekaterinakokatjuhha'],
  'ids': {'anchor': 'The-Journey-of-a-Data-Generalist-From-Bioinformatics-to-Freelancing---Jekaterina-Kokatjuhha-e1upvim',
   'youtube': 'FRi0SUtxdMw'},
  'image': 'images/podcast/s12e08-journey-of-data-generalist-from-bioinformatics-to-freelancing.jpg',
  'links': {'anchor': 'https://anchor.fm/datatalksclub/episodes/The-Journey-of-a-Data-Generalist-From-Bioinformatics-to-Freelancing---Jekaterina-Kokatjuhha-e1upvim',
   'apple': 'https://podcasts.apple.com/us/podcast/the-journey-of-a-data-generalist-from/id1541710331?i=1000599125044',
   'spotify': 'https://open.spotify.com/episode/5fB185hGlGYQmdk0kbIsPv?si=YtnsaYNzTc-fl7emZ2IjEA',
   'youtube': 'https://www.youtube.com/watch?v=FRi0SUtxdMw'},
  'season': 12,
  'short': 'The Journey of a Data Generalist: From Bioinformatics to Freelancing',
  'title': 'The Journey of a Data Generalist: From Bioinformatics to Freelancing',
  'transcript': [{'line': "This week we'll talk about being a 

In [232]:
# Chunking podcast data
lines = [
    entry["line"]
    for ep in with_youtube
    for entry in (ep.get("transcript") or [])
    if isinstance(entry, dict) and "line" in entry
]

In [233]:
from typing import List, Dict, Any, Tuple

def extract_transcript_time_text(episodes: List[Dict[str, Any]]) -> List[Tuple[str, str]]:
    """
    Extracts (time, text) pairs from all podcast transcripts.
    """
    data: List[Tuple[str, str]] = []

    for ep in episodes:
        transcript = ep.get("transcript") or []
        if not isinstance(transcript, list):
            continue

        for entry in transcript:
            if isinstance(entry, dict) and "line" in entry:
                time = entry.get("time", None)
                text = entry["line"]
                data.append((time, text))
    
    return data

In [234]:
with_youtube_lines = extract_transcript_time_text(with_youtube)
with_youtube_lines


[('1:11',
  "This week we'll talk about being a data generalist. We'll discuss going from bioinformatics to freelancing. We have a special guest today, Katya. As a freelancer Katya is helping companies bridge the gap between business and data by building actionable analytics and coaching the teams. She has a lot of broad experience in startups, entrepreneurship and scale-ups. Katya was head of analytics at Gitti, a beauty brand. She tried to start her own fintech business with Entrepreneur First and she worked as a data scientist at Zalando. Welcome to the show. It's a pleasure to have you here."),
 ('1:52',
  "Yes, thank you so much for the invitation. It was really nice to catch up, actually. I think we've known each other for some time. I'm really happy to be here."),
 ('2:02',
  "I tried to invite you multiple times. Finally, we managed to do this. [chuckles] Before we start with our main topic of being a data generalist, let's start with your background. Can you tell us about your

In [235]:
print(with_youtube_lines[0:5])

[('1:11', "This week we'll talk about being a data generalist. We'll discuss going from bioinformatics to freelancing. We have a special guest today, Katya. As a freelancer Katya is helping companies bridge the gap between business and data by building actionable analytics and coaching the teams. She has a lot of broad experience in startups, entrepreneurship and scale-ups. Katya was head of analytics at Gitti, a beauty brand. She tried to start her own fintech business with Entrepreneur First and she worked as a data scientist at Zalando. Welcome to the show. It's a pleasure to have you here."), ('1:52', "Yes, thank you so much for the invitation. It was really nice to catch up, actually. I think we've known each other for some time. I'm really happy to be here."), ('2:02', "I tried to invite you multiple times. Finally, we managed to do this. [chuckles] Before we start with our main topic of being a data generalist, let's start with your background. Can you tell us about your career 

In [236]:
def sliding_window(seq, size, step):
    """Create overlapping chunks using sliding window approach."""
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size]
        result.append(batch)
        if i + size >= n:
            break

    return result


In [237]:

chunks = [" ".join(chunk) for chunk in sliding_window(lines, size=30, step=15)]
print(f"Total chunks: {len(chunks)}")

Total chunks: 1577


In [238]:
chunks[0]

"This week we'll talk about being a data generalist. We'll discuss going from bioinformatics to freelancing. We have a special guest today, Katya. As a freelancer Katya is helping companies bridge the gap between business and data by building actionable analytics and coaching the teams. She has a lot of broad experience in startups, entrepreneurship and scale-ups. Katya was head of analytics at Gitti, a beauty brand. She tried to start her own fintech business with Entrepreneur First and she worked as a data scientist at Zalando. Welcome to the show. It's a pleasure to have you here. Yes, thank you so much for the invitation. It was really nice to catch up, actually. I think we've known each other for some time. I'm really happy to be here. I tried to invite you multiple times. Finally, we managed to do this. [chuckles] Before we start with our main topic of being a data generalist, let's start with your background. Can you tell us about your career journey so far? Yes. Let me start a 

In [239]:
# index the data 

from minsearch import Index

def sliding_window(
        seq: Iterable[Any],
        size: int,
        step: int
    ) -> List[Dict[str, Any]]:
    """
    Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size (int): The size of each chunk/window.
        step (int): The step size between consecutive windows.

    Returns:
        list: A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size]
        result.append({'start': i, 'content': batch})
        if i + size > n:
            break

    return result


def chunk_documents(
        documents: Iterable[Dict[str, str]],
        size: int = 2000,
        step: int = 1000,
        content_field_name: str = 'content'
) -> List[Dict[str, str]]:
    """
    Split a collection of documents into smaller chunks using sliding windows.

    Takes documents and breaks their content into overlapping chunks while preserving
    all other document metadata (filename, etc.) in each chunk.

    Args:
        documents: An iterable of document dictionaries. Each document must have a content field.
        size (int, optional): The maximum size of each chunk. Defaults to 2000.
        step (int, optional): The step size between chunks. Defaults to 1000.
        content_field_name (str, optional): The name of the field containing document content.
                                          Defaults to 'content'.

    Returns:
        list: A list of chunk dictionaries. Each chunk contains:
            - All original document fields except the content field
            - 'start': Starting position of the chunk in original content
            - 'content': The chunk content

    Example:
        >>> documents = [{'content': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, size=100, step=50)
        >>> # Or with custom content field:
        >>> documents = [{'text': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, content_field_name='text')
    """
    results = []

    for doc in documents:
        doc_copy = doc.copy()
        doc_content = doc_copy.pop(content_field_name)
        chunks = sliding_window(doc_content, size=size, step=step)
        for chunk in chunks:
            chunk.update(doc_copy)
        results.extend(chunks)

    return results





def index_documents(documents, chunk: bool = False, chunking_params=None) -> Index:
    """
    Create a searchable index from a collection of documents.

    Args:
        documents: A collection of document dictionaries, each containing at least
                  'content' and 'filename' fields.
        chunk (bool, optional): Whether to chunk documents before indexing.
                               Defaults to False.
        chunking_params (dict, optional): Parameters for document chunking.
                                        Defaults to {'size': 2000, 'step': 1000}.
                                        Only used when chunk=True.

    Returns:
        Index: A fitted minsearch Index object ready for searching.

    Example:
        >>> docs = [{'content': 'Hello world', 'filename': 'doc1.txt'}]
        >>> index = index_documents(docs)
        >>> results = index.search('hello')
    """
    if chunk:
        if chunking_params is None:
            chunking_params = {'size': 2000, 'step': 1000}
        documents = chunk_documents(documents, **chunking_params)

    index = Index(
        text_fields=["content"],
    )

    index.fit(documents)
    return index



In [240]:
documents = [
    {"content": chunk, }
    for i, chunk in enumerate(chunks, start=1)
]


In [241]:
documents[0]

{'content': "This week we'll talk about being a data generalist. We'll discuss going from bioinformatics to freelancing. We have a special guest today, Katya. As a freelancer Katya is helping companies bridge the gap between business and data by building actionable analytics and coaching the teams. She has a lot of broad experience in startups, entrepreneurship and scale-ups. Katya was head of analytics at Gitti, a beauty brand. She tried to start her own fintech business with Entrepreneur First and she worked as a data scientist at Zalando. Welcome to the show. It's a pleasure to have you here. Yes, thank you so much for the invitation. It was really nice to catch up, actually. I think we've known each other for some time. I'm really happy to be here. I tried to invite you multiple times. Finally, we managed to do this. [chuckles] Before we start with our main topic of being a data generalist, let's start with your background. Can you tell us about your career journey so far? Yes. Let

In [242]:
index = index_documents(documents)

In [243]:
index.fit(documents)

In [244]:
index.search('how do I make money with AI?')

[{'content': '23:21 Software engineering practices in particle physics 26:11 Challenges during interviews for data science roles 29:30 Mentoring and offering advice to job seekers 40:08 The STAR method and its value in interviews 50:32 Paid vs unpaid mentorship and finding the right fit This week, we’re discussing the practical application of generative AI in industry. Our special guest today is Maria, a Principal Key Expert in Artificial Intelligence at Siemens. Maria has over 15 years of experience in AI and has a reputation for transforming advanced AI research into impactful, practical tools. Her work focuses on creating scalable solutions that enhance decision-making and streamline processes. Also, for anyone only listening to the audio version, you might want to check out the video because we just saw a cute cat! Welcome, Maria! Thanks to Johanna Bayer for preparing today’s questions. Let’s start with your background. Can you tell us about your career journey so far? Thank you fo

Lessons from first week on the Agentic Course at DataTalksClub. All RAG applications follow a similar structure based on three key steps: searching, building the right prompt and applying an LLM model. My key learnings from this week:

- The searching process seems to be the most critical step (at least where I struggled more). A well parsed dataset and a clear chunking strategy may make your life way easier. 

- Having a well structured prompt with the right context not only improves the performance of your RAG but also make it more efficient from the cost side. 

Looking forward to 



Lessons from the first week at the AI Bootcamp from Alexey Grigorev. This week has addressed the foundations of RAG applications. All RAG follow a similar structure built around three key steps: searching, building the right prompt, and applying an LLM model.

My key learnings from this week:

The searching process seems to be the most critical step (and definitely the one I struggled with the most). A well-parsed dataset and a clear chunking strategy can make your life much easier.

Having a well-structured prompt with the right context not only improves your RAG’s performance but also makes it more cost-efficient.